# LLM Agent / Tool / Retrieval 디버그 플레이북

이 노트북은 지금 단계에서 필요한 모듈 경계를 직접 실행해 보며 확인하기 위한 디버깅용 노트북입니다.

## 목적

- Ollama 로컬 모델 호출이 가능한지 확인합니다.
- 구조화 출력 스키마가 실제 모델 응답을 검증할 수 있는지 확인합니다.
- Qdrant 기반 retrieval service의 최소 동작을 확인합니다.
- retrieval service를 LangChain tool로 감싼 뒤 직접 호출해 봅니다.
- LangChain agent가 retrieval tool을 선택하고 호출할 수 있는지 smoke test합니다.

## 중요한 전제

아직 `app/` 실제 모듈이 구현된 상태가 아니므로, 이 노트북은 앞으로 만들 모듈을 작은 프로토타입으로 재현합니다. 여기서 안정적인 형태가 확인되면 다음 파일 구조로 옮기면 됩니다.

```text
app/
  agents/email_analysis_agent.py
  tools/retrieval_tools.py
  services/retrieval_service.py
  repositories/qdrant_repository.py
  schemas/email_analysis.py
  schemas/retrieval.py
  llm/models.py
```

## 지금 단계에서 이른 것

- 복잡한 multi-step agentic RAG
- LangGraph checkpoint 기반 장기 실행 workflow
- agent가 여러 검색 tool을 자유롭게 반복 호출하는 운영 흐름
- 검색 결과를 근거 없이 최종 분류에 강하게 반영하는 구조

## 지금 단계에서 확인할 것

- 기본 LLM 분석이 schema validation을 통과하는가
- Qdrant 검색 결과가 원본 이메일 ID와 연결되는가
- retrieval tool의 입력/출력 모양이 agent가 쓰기 쉬운가
- LangChain trace/eval로 관찰 가능한 agent/tool 경계가 만들어지는가

## 1. 실행 환경과 패키지 확인

이 셀은 현재 Python 환경에 필요한 패키지가 있는지 확인합니다.

필수에 가까운 패키지:

- `requests`
- `pydantic`
- `qdrant-client`
- `langchain`
- `langchain-ollama`

선택 패키지:

- `openai`: OpenAI-compatible endpoint 호출 실험에 사용
- `langsmith`: LLMOps trace/eval 실험에 사용

In [ ]:
import importlib.util
import json
import os
import time
import uuid
from dataclasses import dataclass
from typing import Any, Literal

import requests
from pydantic import BaseModel, Field, ValidationError


def package_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


packages = {
    "requests": package_available("requests"),
    "pydantic": package_available("pydantic"),
    "openai": package_available("openai"),
    "qdrant_client": package_available("qdrant_client"),
    "langchain": package_available("langchain"),
    "langchain_core": package_available("langchain_core"),
    "langchain_ollama": package_available("langchain_ollama"),
    "langsmith": package_available("langsmith"),
}

packages

## 2. 공통 설정

환경변수로 기본값을 바꿀 수 있습니다.

| 환경변수 | 기본값 | 용도 |
|---|---|---|
| `OLLAMA_BASE_URL` | `http://localhost:11434` | Ollama native API |
| `OLLAMA_OPENAI_BASE_URL` | `http://localhost:11434/v1` | OpenAI-compatible API |
| `OLLAMA_MODEL` | `llama3.2:latest` | agent/analysis 모델 |
| `EMBEDDING_MODEL` | `bge-m3:latest` | 검색용 임베딩 모델 |
| `EMBEDDING_VECTOR_SIZE` | `1024` | Qdrant 벡터 차원 |
| `QDRANT_URL` | 없음 | 서버형 Qdrant URL |
| `QDRANT_PATH` | `/tmp/coramail_agent_qdrant_debug` | 로컬 파일형 Qdrant path |

기본 `QDRANT_PATH`는 `/tmp`를 사용합니다. 기존 repo의 `data/qdrant`를 건드리지 않기 위한 선택입니다.

In [ ]:
@dataclass(frozen=True)
class DebugSettings:
    ollama_base_url: str = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")
    ollama_openai_base_url: str = os.getenv("OLLAMA_OPENAI_BASE_URL", "http://localhost:11434/v1").rstrip("/")
    analysis_model: str = os.getenv("OLLAMA_MODEL", "llama3.2:latest")
    embedding_model: str = os.getenv("EMBEDDING_MODEL", "bge-m3:latest")
    embedding_vector_size: int = int(os.getenv("EMBEDDING_VECTOR_SIZE", "1024"))
    qdrant_url: str | None = os.getenv("QDRANT_URL") or None
    qdrant_path: str = os.getenv("QDRANT_PATH", "/tmp/coramail_agent_qdrant_debug")
    qdrant_collection: str = os.getenv("QDRANT_DEBUG_COLLECTION", "coramail_debug_emails")


settings = DebugSettings()
settings

## 3. Ollama 상태 확인

여기서 확인할 것:

- Ollama 서버가 떠 있는가
- 설치된 모델 목록에 `analysis_model`과 `embedding_model`이 있는가
- 모델명이 틀렸을 때 어디서 실패하는지 명확히 보이는가

In [ ]:
def get_ollama_models() -> list[str]:
    response = requests.get(f"{settings.ollama_base_url}/api/tags", timeout=10)
    response.raise_for_status()
    return [model["name"] for model in response.json().get("models", [])]


def resolve_ollama_model_name(model_name: str, available_models: list[str]) -> str | None:
    """Ollama 모델명은 `name`이 `bge-m3:latest`처럼 tag를 포함할 수 있습니다."""
    if model_name in available_models:
        return model_name

    latest_name = f"{model_name}:latest"
    if latest_name in available_models:
        return latest_name

    return None


try:
    root_response = requests.get(f"{settings.ollama_base_url}/", timeout=10)
    print("ollama_root:", root_response.status_code, root_response.text[:100])

    available_models = get_ollama_models()
    resolved_analysis_model = resolve_ollama_model_name(settings.analysis_model, available_models)
    resolved_embedding_model = resolve_ollama_model_name(settings.embedding_model, available_models)
    print("available_models:", available_models)
    print("analysis_model:", settings.analysis_model, "->", resolved_analysis_model)
    print("embedding_model:", settings.embedding_model, "->", resolved_embedding_model)
    print("analysis_model_ok:", resolved_analysis_model is not None)
    print("embedding_model_ok:", resolved_embedding_model is not None)
except Exception as exc:
    print("Ollama 확인 실패:", repr(exc))
    print("확인: ollama serve 또는 Ollama 앱이 실행 중인지 확인하세요.")

## 4. 공통 스키마

이 스키마는 나중에 `app/schemas/`로 옮길 후보입니다.

핵심 원칙:

- LLM이 반환하는 값은 무조건 Pydantic으로 검증합니다.
- agent/tool 내부 포맷과 DB 저장 포맷을 분리할 수 있게 class를 작게 유지합니다.
- enum 값은 내부 코드이므로 영어를 유지하고, 화면 표시만 한글 매핑을 둡니다.

In [ ]:
class EmailInput(BaseModel):
    email_uid: str
    subject: str
    sender: str
    body: str
    attachments: list[str] = Field(default_factory=list)


class EmailAnalysisResult(BaseModel):
    # predicted_email_intent 라벨은 임시 업무 분류 체계입니다. 실제 운영 전 반드시 사용자 검토가 필요합니다.
    # - inquiry: 견적/납기/제품 문의처럼 아직 발주가 확정되지 않은 요청
    # - order: 구매 발주서, PO, 주문 확정처럼 실제 주문 처리로 이어지는 요청
    # - service: 클레임, 고장, 누수, 긴급 지원처럼 서비스/AS 대응이 필요한 요청
    # - technical: 도면, 사양, 기술 검토, 호환성 확인처럼 엔지니어링 판단이 필요한 요청
    # - other: 위 기준으로 분류하기 어렵거나 추가 업무 유형 정의가 필요한 메일
    # TODO: 실제 고객 메일 샘플을 보고 라벨 이름, 개수, 정의를 확정해야 합니다.
    predicted_email_intent: Literal["inquiry", "order", "service", "technical", "other"]
    # predicted_email_importance 라벨도 임시 우선순위 체계입니다. SLA, 업무 프로세스, 사용자 화면 정책에 맞게 조정해야 합니다.
    # - low: 참고/일반 정보성으로 즉시 처리가 필요하지 않은 메일
    # - normal: 통상 처리 기한 안에 대응하면 되는 일반 업무 메일
    # - high: 납기, 견적 마감, 고객 영향 등으로 우선 확인이 필요한 메일
    # - urgent: 긴급 수리, 선박 운항 영향, 즉시 회신 요구처럼 지연 시 손실이 큰 메일
    # TODO: predicted_email_importance는 단순 감정/단어가 아니라 실제 처리 SLA와 연결해 정의해야 합니다.
    predicted_email_importance: Literal["low", "normal", "high", "urgent"]
    # generated_summary는 사용자가 메일 목록/상세에서 빠르게 이해하기 위한 한두 문장 요약입니다.
    # TODO: 화면 길이와 업무 용어에 맞게 최대 길이, 금지 표현, 포함 필드를 정해야 합니다.
    generated_summary: str
    # extracted_key_information은 모델이 찾은 업무 핵심 필드를 담는 임시 dict입니다.
    # TODO: 견적번호, PO 번호, 선박명, 제품명, 수량, 납기일 등 확정 필드는 별도 Pydantic 모델로 승격해야 합니다.
    extracted_key_information: dict[str, Any] = Field(default_factory=dict)
    # predicted_assignee_area는 실제 개인 담당자라기보다 임시 담당 영역/팀 후보입니다.
    # 예: sales_team, service_team, technical_team, order_management 등.
    # TODO: 실제 사용자/팀/라우팅 규칙이 정리되면 assignee_user_id, assignee_team_id와 분리할지 결정해야 합니다.
    predicted_assignee_area: str
    # prediction_reasoning은 위 예측값을 낸 근거 설명입니다.
    # TODO: 실제 UI에 노출할지, 내부 감사/디버깅 용도로만 저장할지 결정해야 합니다.
    prediction_reasoning: str
    # predicted_needs_human_review는 AI가 사람 검토 필요성을 예측한 값입니다.
    # 최종 검토 상태가 아니며, 서비스 계층에서 정책/신뢰도/오류 여부와 함께 확정해야 합니다.
    predicted_needs_human_review: bool


class RetrievalResult(BaseModel):
    source_type: Literal["email", "attachment_chunk"]
    source_id: str
    score: float
    title: str
    snippet: str
    payload: dict[str, Any] = Field(default_factory=dict)


sample_email = EmailInput(
    email_uid="debug-email-001",
    subject="펌프 예비품 견적 요청",
    sender="customer@example.com",
    body="선박 정기 수리에 사용할 펌프 예비품 견적을 요청드립니다. 가격과 납기일을 긴급히 회신 부탁드립니다.",
    attachments=["펌프_예비품_RFQ.pdf"],
)

sample_email

## 5. LLM 호출 모듈 프로토타입

나중에 `app/llm/client.py`로 옮길 후보입니다.

이 셀은 두 방식을 지원합니다.

1. `openai` 패키지가 있으면 Ollama OpenAI-compatible endpoint 사용
2. 없으면 Ollama native `/api/chat` 직접 호출

LangChain agent를 도입하더라도, 단일 구조화 분석 smoke test는 이런 작은 함수로 계속 유지하는 것이 좋습니다. agent가 실패했을 때 모델 자체 문제인지 agent/tool loop 문제인지 분리해서 볼 수 있기 때문입니다.

In [ ]:
def build_analysis_prompt(email: EmailInput) -> str:
    return f"""
선박 부품 제조사 업무 이메일을 분석하세요.
반드시 아래 JSON 스키마에 맞는 JSON만 반환하세요.

스키마:
{json.dumps(EmailAnalysisResult.model_json_schema(), ensure_ascii=False, indent=2)}

이메일:
{email.model_dump_json(indent=2)}
"""


def generate_structured_analysis(email: EmailInput) -> EmailAnalysisResult:
    prompt = build_analysis_prompt(email)

    if packages.get("openai"):
        from openai import OpenAI

        client = OpenAI(base_url=settings.ollama_openai_base_url, api_key="ollama")
        completion = client.chat.completions.create(
            model=settings.analysis_model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"},
        )
        content = completion.choices[0].message.content
    else:
        payload = {
            "model": settings.analysis_model,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "format": "json",
            "options": {"temperature": 0},
        }
        response = requests.post(f"{settings.ollama_base_url}/api/chat", json=payload, timeout=120)
        response.raise_for_status()
        content = response.json()["message"]["content"]

    print("raw_llm_output:")
    print(content)
    return EmailAnalysisResult.model_validate_json(content)

In [ ]:
# 단일 LLM 분석 smoke test입니다.
# 실패하면 agent/tool 이전에 모델 호출, 모델명, JSON 출력 품질부터 확인하세요.
try:
    analysis_result = generate_structured_analysis(sample_email)
    analysis_result
except Exception as exc:
    print("LLM 분석 실패:", type(exc).__name__, exc)

## 6. Qdrant Repository / Retrieval Service 프로토타입

이 섹션은 나중에 다음 모듈로 분리할 후보입니다.

```text
app/repositories/qdrant_repository.py
app/services/retrieval_service.py
```

지금은 운영 컬렉션을 건드리지 않기 위해 기본적으로 `/tmp/coramail_agent_qdrant_debug`에 디버그용 Qdrant local storage를 만듭니다.

여기서 확인할 것:

- 임베딩 모델이 정상 동작하는가
- Qdrant 컬렉션 생성과 upsert가 되는가
- 검색 결과가 `RetrievalResult` 스키마로 정리되는가

In [ ]:
if not packages.get("qdrant_client"):
    raise RuntimeError("qdrant-client가 필요합니다. 현재 환경에 설치되어 있지 않습니다.")

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams


def create_qdrant_client() -> QdrantClient:
    if settings.qdrant_url:
        return QdrantClient(
            url=settings.qdrant_url,
            api_key=os.getenv("QDRANT_API_KEY") or None,
            timeout=30,
        )
    return QdrantClient(path=settings.qdrant_path)


qdrant_client = create_qdrant_client()
print(qdrant_client)
print(qdrant_client.get_collections())

In [ ]:
def embed_text(text: str) -> list[float]:
    # 최신 Ollama는 /api/embed를 사용합니다.
    model_name = globals().get("resolved_embedding_model") or settings.embedding_model

    response = requests.post(
        f"{settings.ollama_base_url}/api/embed",
        json={"model": model_name, "input": text},
        timeout=120,
    )

    # 일부 구버전/모델 조합에서는 /api/embeddings만 동작할 수 있어 fallback을 둡니다.
    if response.status_code == 404:
        response = requests.post(
            f"{settings.ollama_base_url}/api/embeddings",
            json={"model": model_name, "prompt": text},
            timeout=120,
        )
        response.raise_for_status()
        return response.json()["embedding"]

    response.raise_for_status()
    data = response.json()
    embeddings = data.get("embeddings")
    if embeddings:
        return embeddings[0]
    return data["embedding"]


test_vector = embed_text("검색 임베딩 테스트")
print("vector_size:", len(test_vector))
print("expected_size:", settings.embedding_vector_size)

In [ ]:
def ensure_debug_collection() -> None:
    if not qdrant_client.collection_exists(settings.qdrant_collection):
        qdrant_client.create_collection(
            collection_name=settings.qdrant_collection,
            vectors_config=VectorParams(
                size=settings.embedding_vector_size,
                distance=Distance.COSINE,
            ),
        )


debug_documents = [
    {
        "source_id": "debug-email-001",
        "title": "펌프 예비품 견적 요청",
        "text": "고객이 선박 정기 수리에 사용할 펌프 예비품 견적과 긴급 납기일 회신을 요청했습니다.",
        "payload": {"source_type": "email", "mail_category": "inquiry", "sender_address": "customer@example.com"},
    },
    {
        "source_id": "debug-email-002",
        "title": "밸브 누수 긴급 서비스 요청",
        "text": "검사 중 밸브 누수가 확인되어 교체 부품과 긴급 서비스 지원 요청이 접수되었습니다.",
        "payload": {"source_type": "email", "mail_category": "service", "sender_address": "shipyard@example.com"},
    },
    {
        "source_id": "debug-email-003",
        "title": "구매 발주서 접수 확인",
        "text": "선박 예비품 구매 발주서 PO-2026-071 접수가 확인되었고 예상 납기 회신이 필요합니다.",
        "payload": {"source_type": "email", "mail_category": "order", "sender_address": "buyer@example.com"},
    },
]


def stable_uuid(text: str) -> str:
    return str(uuid.uuid5(uuid.NAMESPACE_URL, text))


def seed_debug_points() -> None:
    ensure_debug_collection()
    points = []
    for doc in debug_documents:
        vector = embed_text(f"{doc['title']}\n{doc['text']}")
        points.append(
            PointStruct(
                id=stable_uuid(doc["source_id"]),
                vector=vector,
                payload={
                    "schema_version": 1,
                    "source_id": doc["source_id"],
                    "title": doc["title"],
                    "snippet": doc["text"],
                    **doc["payload"],
                },
            )
        )
    qdrant_client.upsert(collection_name=settings.qdrant_collection, points=points)


seed_debug_points()
print(qdrant_client.get_collection(settings.qdrant_collection))

In [ ]:
class RetrievalService:
    def __init__(self, client: QdrantClient, collection_name: str):
        self.client = client
        self.collection_name = collection_name

    def search_similar_emails(self, query: str, limit: int = 5) -> list[RetrievalResult]:
        query_vector = embed_text(query)

        # qdrant-client 버전에 따라 query_points 또는 search를 사용합니다.
        if hasattr(self.client, "query_points"):
            response = self.client.query_points(
                collection_name=self.collection_name,
                query=query_vector,
                limit=limit,
                with_payload=True,
            )
            points = response.points
        else:
            points = self.client.search(
                collection_name=self.collection_name,
                query_vector=query_vector,
                limit=limit,
                with_payload=True,
            )

        results = []
        for point in points:
            payload = point.payload or {}
            results.append(
                RetrievalResult(
                    source_type=payload.get("source_type", "email"),
                    source_id=payload.get("source_id", str(point.id)),
                    score=float(point.score),
                    title=payload.get("title", ""),
                    snippet=payload.get("snippet", ""),
                    payload=payload,
                )
            )
        return results


retrieval_service = RetrievalService(qdrant_client, settings.qdrant_collection)
retrieval_results = retrieval_service.search_similar_emails("밸브 누수 긴급 서비스 지원", limit=3)
[result.model_dump() for result in retrieval_results]

## 7. LangChain Tool 프로토타입

이 섹션은 나중에 `app/tools/retrieval_tools.py`로 옮길 후보입니다.

원칙:

- tool 함수는 얇게 둡니다.
- tool 안에서 Qdrant를 직접 다루지 않고 `retrieval_service`를 호출합니다.
- 반환값은 agent와 trace에서 읽기 쉽게 JSON 문자열로 둡니다.
- 처음에는 read-only tool만 허용합니다.

In [ ]:
if not packages.get("langchain_core"):
    raise RuntimeError("langchain-core 또는 langchain이 필요합니다. 현재 환경에 설치되어 있지 않습니다.")

try:
    from langchain_core.tools import tool
except ImportError:
    from langchain.tools import tool


@tool
def search_similar_emails(query: str, limit: int = 3) -> str:
    """유사한 과거 이메일을 검색합니다.

    Args:
        query: 검색할 자연어 질문 또는 이메일 요약.
        limit: 반환할 검색 결과 수.
    """
    results = retrieval_service.search_similar_emails(query=query, limit=limit)
    return json.dumps([result.model_dump() for result in results], ensure_ascii=False, indent=2)


tools = [search_similar_emails]
tools

In [ ]:
# agent에 붙이기 전에 tool 단독 실행부터 확인합니다.
print(search_similar_emails.invoke({"query": "펌프 예비품 견적과 납기일", "limit": 2}))

## 8. LangChain Agent Smoke Test

이 섹션은 나중에 `app/agents/email_analysis_agent.py`로 옮길 후보입니다.

여기서 확인할 것:

- agent가 retrieval tool을 호출하는가
- tool 입력 argument가 정상인가
- tool 결과를 최종 답변에 반영하는가
- LangSmith/Langfuse trace에서 model call과 tool call이 분리되어 보이는가

주의:

- 로컬 Ollama 모델이 tool calling을 안정적으로 지원하지 않으면 이 셀이 실패하거나 tool을 호출하지 않을 수 있습니다.
- 실패하더라도 구조가 틀렸다고 단정하지 말고, 모델의 tool-call 지원 품질을 별도 이슈로 분리하세요.

In [ ]:
if not packages.get("langchain") or not packages.get("langchain_ollama"):
    raise RuntimeError("langchain과 langchain-ollama가 필요합니다. 현재 환경에 설치되어 있지 않습니다.")

from langchain.agents import create_agent
from langchain_ollama import ChatOllama


agent_model = ChatOllama(
    model=settings.analysis_model,
    base_url=settings.ollama_base_url,
    temperature=0,
)

system_prompt = """
당신은 선박 부품 제조사의 이메일 분석 agent입니다.
사용자가 유사 과거 사례나 문맥 확인을 요구하면 search_similar_emails tool을 사용하세요.
tool 결과를 그대로 복사하지 말고, 어떤 과거 이메일이 관련 있는지 간결히 설명하세요.
"""

agent = create_agent(
    model=agent_model,
    tools=tools,
    system_prompt=system_prompt,
)

agent

In [ ]:
agent_input = {
    "messages": [
        {
            "role": "user",
            "content": "밸브 누수 긴급 서비스 요청과 유사한 과거 이메일을 찾아서 어떤 건이 관련 있는지 알려줘.",
        }
    ]
}

try:
    agent_result = agent.invoke(agent_input)
    agent_result
except Exception as exc:
    print("Agent 실행 실패:", type(exc).__name__, exc)
    print("확인할 것: 모델 tool calling 지원, langchain/langchain-ollama 버전, Ollama 모델명")

## 9. LLMOps 확인 포인트

LangChain을 최소 도입하는 이유는 단순히 코드가 짧아서가 아니라, agent/tool workflow를 관찰하고 평가하기 쉽기 때문입니다.

### LangSmith를 쓸 경우

환경변수:

```bash
export LANGSMITH_TRACING=true
export LANGSMITH_API_KEY=<key>
export LANGSMITH_PROJECT=coramail-agent-debug
```

확인할 trace:

- user input
- model call
- tool call name
- tool arguments
- tool result
- final answer
- latency
- error

### Langfuse/OpenTelemetry를 쓸 경우

LangChain/LangGraph integration 또는 OpenTelemetry exporter를 사용합니다. 선택과 무관하게 우리 DB에도 최소 실행 로그를 남겨야 합니다.

우리 DB에 남길 최소 필드 후보:

- `run_id`
- `email_uid`
- `agent_name`
- `prompt_name`
- `prompt_version`
- `model`
- `input_hash`
- `output_json`
- `schema_validation_status`
- `tool_calls`
- `latency_ms`
- `error_type`
- `created_at`

In [ ]:
debug_checklist = {
    "ollama_server": "3번 셀에서 /api/tags 성공",
    "analysis_model": settings.analysis_model,
    "embedding_model": settings.embedding_model,
    "schema_validation": "5번 LLM 분석 smoke test 통과",
    "qdrant_collection": settings.qdrant_collection,
    "retrieval_service": "6번 search_similar_emails 결과 확인",
    "retrieval_tool": "7번 tool 단독 실행 결과 확인",
    "agent_tool_call": "8번 agent가 search_similar_emails를 호출하는지 확인",
    "llmops_trace": "LangSmith/Langfuse에서 model/tool call 분리 확인",
}

debug_checklist

## 10. 이 노트북에서 확인 후 다음 구현 순서

1. `schemas/`에 `EmailInput`, `EmailAnalysisResult`, `RetrievalResult`를 만든다.
2. `repositories/qdrant_repository.py`에 Qdrant 접근 코드를 옮긴다.
3. `services/retrieval_service.py`에 검색 정책과 결과 정규화를 옮긴다.
4. `tools/retrieval_tools.py`에 LangChain tool wrapper를 만든다.
5. `agents/email_analysis_agent.py`에서 LangChain `create_agent`를 사용한다.
6. `services/email_analysis_service.py`는 agent 실행 결과 검증, 저장, 사람 검토 상태 전이를 담당한다.
7. LLMOps trace와 별도로 PostgreSQL에도 최소 실행 로그를 저장한다.

엄밀한 순서로 가려면, agentic RAG를 바로 운영 흐름에 넣기보다 이 노트북에서 다음 3가지를 먼저 통과시키세요.

- retrieval service 단독 검색 품질
- retrieval tool 단독 입출력 안정성
- agent가 tool을 호출하는지에 대한 trace 확인